# 02 — Cleaning & Feature Engineering: OWID CO₂ Dataset

Standardize data, engineer analytical features, segment countries, and export processed datasets for EDA and modelling.

In [1]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

PROJECT_ROOT  = Path("..").resolve()
RAW_PATH      = PROJECT_ROOT / "data" / "raw" / "owid-co2-data.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AGG_DIR       = PROJECT_ROOT / "data" / "aggregated"
FIGURE_DIR    = PROJECT_ROOT / "reports" / "figures"
for p in [PROCESSED_DIR, AGG_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def section(t):
    print("\n" + "="*90); print(t); print("="*90)


In [2]:
section("Load raw data & separate countries from aggregate regions")

df_raw = pd.read_csv(RAW_PATH)

# Individual countries have an ISO code; aggregate regions do not
df = df_raw[df_raw["iso_code"].notna()].copy()
df.reset_index(drop=True, inplace=True)

print(f"Raw rows:     {len(df_raw):,}")
print(f"Country rows: {len(df):,}")
print(f"Unique countries: {df['country'].nunique()}")
display(df.head())



Load raw data & separate countries from aggregate regions
Raw rows:     50,411
Country rows: 42,480
Unique countries: 218


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,co2_including_luc,co2_including_luc_growth_abs,co2_including_luc_growth_prct,co2_including_luc_per_capita,co2_including_luc_per_gdp,co2_including_luc_per_unit_energy,co2_per_capita,co2_per_gdp,co2_per_unit_energy,coal_co2,coal_co2_per_capita,consumption_co2,consumption_co2_per_capita,consumption_co2_per_gdp,cumulative_cement_co2,cumulative_co2,cumulative_co2_including_luc,cumulative_coal_co2,cumulative_flaring_co2,cumulative_gas_co2,cumulative_luc_co2,cumulative_oil_co2,cumulative_other_co2,energy_per_capita,energy_per_gdp,flaring_co2,flaring_co2_per_capita,gas_co2,gas_co2_per_capita,ghg_excluding_lucf_per_capita,ghg_per_capita,land_use_change_co2,land_use_change_co2_per_capita,methane,methane_per_capita,nitrous_oxide,nitrous_oxide_per_capita,oil_co2,oil_co2_per_capita,other_co2_per_capita,other_industry_co2,primary_energy_consumption,share_global_cement_co2,share_global_co2,share_global_co2_including_luc,share_global_coal_co2,share_global_cumulative_cement_co2,share_global_cumulative_co2,share_global_cumulative_co2_including_luc,share_global_cumulative_coal_co2,share_global_cumulative_flaring_co2,share_global_cumulative_gas_co2,share_global_cumulative_luc_co2,share_global_cumulative_oil_co2,share_global_cumulative_other_co2,share_global_flaring_co2,share_global_gas_co2,share_global_luc_co2,share_global_oil_co2,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
section("Cleaning — types, range guards, fill strategy")

# Enforce numeric on all non-identifier columns
id_cols = ["country", "iso_code"]
num_cols = [c for c in df.columns if c not in id_cols]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Remove physically impossible values
invalid = (
    (df["co2"] < 0) |
    (df["co2_per_capita"] < 0) |
    (df["population"] < 0) |
    (df["year"] < 1750) |
    (df["year"] > 2030)
)
print(f"Invalid rows removed: {invalid.sum()}")
df = df[~invalid].copy()
print(f"Clean rows: {len(df):,}")



Cleaning — types, range guards, fill strategy
Invalid rows removed: 0
Clean rows: 42,480


In [4]:
section("Feature engineering — emissions intensity & income proxy")

# ── Emission intensity features ───────────────────────────────────────────
df["co2_per_energy"] = np.where(
    df["primary_energy_consumption"] > 0,
    df["co2"] / df["primary_energy_consumption"],
    np.nan
)

df["fossil_share"] = (
    (df[["coal_co2","oil_co2","gas_co2"]].sum(axis=1))
    / df["co2"].replace(0, np.nan)
).clip(0, 1)

df["luc_share"] = (
    df["land_use_change_co2"] / df["co2_including_luc"].replace(0, np.nan)
).clip(-1, 1)

# ── GDP per capita proxy (income level) ───────────────────────────────────
df["gdp_per_capita"] = np.where(
    df["population"] > 0,
    df["gdp"] / df["population"],
    np.nan
)

# ── Annual growth category ─────────────────────────────────────────────────
def growth_bucket(g):
    if pd.isna(g): return "Unknown"
    if g < -5:     return "Sharp decline (>-5%)"
    if g < 0:      return "Decline"
    if g == 0:     return "Flat"
    if g < 5:      return "Moderate growth"
    return "Rapid growth (>5%)"

df["co2_growth_cat"] = df["co2_growth_prct"].apply(growth_bucket)

# ── Income tier (based on gdp_per_capita, updated thresholds) ──────────────
def income_tier(x):
    if pd.isna(x): return "Unknown"
    if x < 1_136:  return "Low income"
    if x < 4_466:  return "Lower-middle income"
    if x < 13_846: return "Upper-middle income"
    return "High income"

df["income_tier"] = df["gdp_per_capita"].apply(income_tier)

# ── Dominant fuel source ───────────────────────────────────────────────────
fuel_cols = ["coal_co2","oil_co2","gas_co2","cement_co2","flaring_co2","land_use_change_co2"]
fuel_cols = [c for c in fuel_cols if c in df.columns]
fuel_labels = {"coal_co2":"Coal","oil_co2":"Oil","gas_co2":"Gas",
               "cement_co2":"Cement","flaring_co2":"Flaring","land_use_change_co2":"LUC"}

def dominant_fuel(row):
    vals = {fuel_labels.get(c, c): row[c] for c in fuel_cols if pd.notna(row[c]) and row[c] > 0}
    return max(vals, key=vals.get) if vals else "Unknown"

df["dominant_fuel"] = df.apply(dominant_fuel, axis=1)

# ── Decarbonisation decade ─────────────────────────────────────────────────
def decade(y):
    d = int(y) // 10 * 10
    return f"{d}s"

df["decade"] = df["year"].apply(decade)

print("Engineered features:")
new_feats = ["co2_per_energy","fossil_share","luc_share","gdp_per_capita",
             "co2_growth_cat","income_tier","dominant_fuel","decade"]
display(df[new_feats].describe(include="all").T)



Feature engineering — emissions intensity & income proxy
Engineered features:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
co2_per_energy,9804.0,NaN,NaN,NaN,0.242149,0.27838,0.0,0.184277,0.219243,0.259709,10.688897
fossil_share,22774.0,NaN,NaN,NaN,0.95374,0.12048,0.0,0.952894,0.995921,1.0,1.0
luc_share,20936.0,NaN,NaN,NaN,0.504379,0.473322,-1.0,0.050363,0.656954,0.949854,1.0
gdp_per_capita,15199.0,NaN,NaN,NaN,8619.534951,11863.324133,361.188725,1869.136291,4114.929621,9942.346265,163531.400281
co2_growth_cat,42480,6,Unknown,20024,NaN,NaN,NaN,NaN,NaN,NaN,NaN
income_tier,42480,5,Unknown,27281,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dominant_fuel,42480,7,LUC,21685,NaN,NaN,NaN,NaN,NaN,NaN,NaN
decade,42480,28,1860s,2180,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
section("Feature dictionary")

feat_dict = pd.DataFrame([
    ["co2_per_energy",   "CO₂ per unit of primary energy consumed — carbon intensity of energy mix"],
    ["fossil_share",     "Share of CO₂ from coal + oil + gas (0–1)"],
    ["luc_share",        "Share of total GHG from land-use change (can be negative for sinks)"],
    ["gdp_per_capita",   "GDP / population — income proxy for wealth segmentation"],
    ["co2_growth_cat",   "Categorical bucket for annual CO₂ growth rate"],
    ["income_tier",      "World Bank-style income grouping based on GDP per capita"],
    ["dominant_fuel",    "Fuel category contributing most CO₂ in that year"],
    ["decade",           "Decade label (1990s, 2000s, …) for temporal grouping"],
], columns=["Feature", "Purpose"])
display(feat_dict)



Feature dictionary


,Feature,Purpose
0,co2_per_energy,CO₂ per unit of primary energy consumed — carb...
1,fossil_share,Share of CO₂ from coal + oil + gas (0–1)
2,luc_share,Share of total GHG from land-use change (can b...
3,gdp_per_capita,GDP / population — income proxy for wealth seg...
4,co2_growth_cat,Categorical bucket for annual CO₂ growth rate
5,income_tier,World Bank-style income grouping based on GDP ...
6,dominant_fuel,Fuel category contributing most CO₂ in that year
7,decade,"Decade label (1990s, 2000s, …) for temporal gr..."


In [6]:
section("Validate engineered features")

print("Income tier distribution:")
display(df["income_tier"].value_counts().to_frame("rows"))
print("\nDominant fuel distribution:")
display(df["dominant_fuel"].value_counts().to_frame("rows"))
print("\nGrowth category distribution:")
display(df["co2_growth_cat"].value_counts().to_frame("rows"))

# Fossil share sanity-check scatter
modern = df[(df["year"] >= 1990) & df["fossil_share"].notna() & df["co2_per_capita"].notna()]
fig = px.scatter(
    modern, x="fossil_share", y="co2_per_capita", color="income_tier",
    opacity=0.5, trendline="ols",
    title="Fossil share vs CO₂ per capita (1990+)",
    labels={"fossil_share": "Fossil share of CO₂", "co2_per_capita": "CO₂ per capita (t)"}
)
fig.update_layout(height=480)
fig.show()



Validate engineered features
Income tier distribution:


,rows
income_tier,
Unknown,27281
Lower-middle income,6566
Upper-middle income,4562
High income,2682
Low income,1389



Dominant fuel distribution:


,rows
dominant_fuel,
LUC,21685
Unknown,9602
Oil,6674
Coal,3554
Gas,833
Flaring,121
Cement,11



Growth category distribution:


,rows
co2_growth_cat,
Unknown,20024
Rapid growth (>5%),10108
Moderate growth,4713
Sharp decline (>-5%),3815
Decline,2888
Flat,932


In [7]:
section("Aggregate tables")

# Focus on modern era for cross-country comparisons
modern = df[df["year"] >= 1990].copy()

# By income tier
income_agg = modern.groupby("income_tier", as_index=False).agg(
    avg_co2_per_cap=("co2_per_capita","mean"),
    avg_fossil_share=("fossil_share","mean"),
    avg_energy_per_cap=("energy_per_capita","mean"),
    avg_gdp_per_cap=("gdp_per_capita","mean"),
    n_rows=("co2","count")
).round(3)
print("By income tier:"); display(income_agg)

# By decade — world totals
world_decade = (
    df[df["country"] == "World"]   # use the World aggregate row
    .groupby("decade", as_index=False)
    .agg(avg_co2=("co2","mean"), max_co2=("co2","max"))
    .round(2)
)
print("\nWorld CO₂ by decade:"); display(world_decade)

# By dominant fuel (country-level, modern)
fuel_agg = modern.groupby("dominant_fuel", as_index=False).agg(
    countries=("country","nunique"),
    avg_co2=("co2","mean"),
    avg_co2_per_cap=("co2_per_capita","mean")
).round(3)
print("\nBy dominant fuel:"); display(fuel_agg)



Aggregate tables
By income tier:


,income_tier,avg_co2_per_cap,avg_fossil_share,avg_energy_per_cap,avg_gdp_per_cap,n_rows
0,High income,10.384,0.944,56896.607,31630.530,1855
1,Low income,0.097,0.925,489.167,856.533,358
2,Lower-middle income,0.959,0.914,2999.433,2390.818,1460
3,Unknown,5.218,0.974,21338.298,NaN,2099
4,Upper-middle income,3.499,0.920,15170.731,8549.112,1736



World CO₂ by decade:


,decade,avg_co2,max_co2



By dominant fuel:


,dominant_fuel,countries,avg_co2,avg_co2_per_cap
0,Coal,43,520.109,7.355
1,Flaring,2,9.071,3.912
2,Gas,35,167.873,11.626
3,LUC,108,30.243,1.069
4,Oil,154,112.084,5.716
5,Unknown,8,0.000,NaN


In [8]:
section("Export processed datasets")

# Full cleaned dataset (country rows only)
df.to_csv(PROCESSED_DIR / "co2_features.csv", index=False)

# Modern era (1990+) subset — main analysis file
modern = df[df["year"] >= 1990].copy()
modern.to_csv(PROCESSED_DIR / "co2_modern.csv", index=False)

# Aggregated tables
income_agg.to_csv(AGG_DIR / "income_tier_summary.csv", index=False)
world_decade.to_csv(AGG_DIR / "world_decade_summary.csv", index=False)
fuel_agg.to_csv(AGG_DIR / "dominant_fuel_summary.csv", index=False)

print("Exported:")
for p in [PROCESSED_DIR, AGG_DIR]:
    for f in sorted(p.iterdir()):
        print(f"  {f}")



Export processed datasets
Exported:
  D:\Data Visualization\project\Group14_IT4023E_Dataviz_project\data\processed\cluster_profiles.csv
  D:\Data Visualization\project\Group14_IT4023E_Dataviz_project\data\processed\cluster_sizes.csv
  D:\Data Visualization\project\Group14_IT4023E_Dataviz_project\data\processed\co2_audit_preview.csv
  D:\Data Visualization\project\Group14_IT4023E_Dataviz_project\data\processed\co2_features.csv
  D:\Data Visualization\project\Group14_IT4023E_Dataviz_project\data\processed\co2_modern.csv
  D:\Data Visualization\project\Group14_IT4023E_Dataviz_project\data\processed\dashboard_concept_map.csv
  D:\Data Visualization\project\Group14_IT4023E_Dataviz_project\data\processed\eda_key_insights.csv
  D:\Data Visualization\project\Group14_IT4023E_Dataviz_project\data\processed\smartphone_audit_preview.csv
  D:\Data Visualization\project\Group14_IT4023E_Dataviz_project\data\processed\smartphone_clustered.csv
  D:\Data Visualization\project\Group14_IT4023E_Dataviz_pr